In [1]:
!pip -q install trafilatura transformers sentence-transformers torch nltk scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 16.2 MB/s eta 0:00:00


In [2]:
# Imports

import trafilatura
import re
import numpy as np
import nltk
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
# 1) Fetch Article

url = "https://apnews.com/article/boeing-aviation-aircraft-air-india-crash-f12b20e65dc57ae655a1e0759b58938f"

downloaded = trafilatura.fetch_url(url)
text = trafilatura.extract(downloaded)

def clean_text(text):
    return re.sub(r'\s+', ' ', text)

text = clean_text(text)

print("Preview:\n", text[:800])


# Split into chunks

from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(text)

def chunk_text(sentences, max_words=150):
    chunks = []
    current = []
    count = 0

    for s in sentences:
        length = len(s.split())
        if count + length <= max_words:
            current.append(s)
            count += length
        else:
            chunks.append(" ".join(current))
            current = [s]
            count = length

    if current:
        chunks.append(" ".join(current))

    return chunks

chunks = chunk_text(sentences)

Preview:
 A look at Boeing’s recent troubles after Air India crash ▶ Follow live updates on the Air India plane crash. The crash of a Boeing 787 passenger jet in India minutes after takeoff on Thursday is putting the spotlight back on a beleaguered manufacturer though it was not immediately clear why the plane crashed. The Air India 787 went down in the northwestern city of Ahmedabad with more than 240 people aboard shortly after takeoff, authorities said. It was the first fatal crash since the plane, also known as the Dreamliner, went into service in 2011, according to the Aviation Safety Network database. Boeing shares fell more than 4% in afternoon trading. The 787 was the first airliner to make extensive use of lithium ion batteries, which are lighter, recharge faster and can hold more energy th


In [4]:
# 2) TRANSFORMER (LLM) ANALYSIS

# Sentiment
sentiment_pipe = pipeline("sentiment-analysis")

def aggregate_sentiment(chunks):
    scores = {"POSITIVE": [], "NEGATIVE": []}

    for c in chunks:
        result = sentiment_pipe(c)[0]
        scores[result['label']].append(result['score'])

    avg = {k: np.mean(v) if v else 0 for k, v in scores.items()}
    label = max(avg, key=avg.get)

    return avg, label

sent_scores, sent_label = aggregate_sentiment(chunks)

# Emotion
emotion_pipe = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base"
)

def aggregate_emotions(chunks):
    emotions = {}

    for c in chunks:
        result = emotion_pipe(c, top_k=None)

        if isinstance(result[0], list):
            result = result[0]

        for r in result:
            emotions.setdefault(r['label'], []).append(r['score'])

    avg = {k: np.mean(v) for k, v in emotions.items()}
    top_emotion = max(avg, key=avg.get)

    return avg, top_emotion

emotion_scores, top_emotion = aggregate_emotions(chunks)

#  Intent (Zero-shot classification)
intent_pipe = pipeline("zero-shot-classification")

candidate_labels = ["informational", "warning", "analytical", "reporting", "opinion"]

intent_result = intent_pipe(text[:1000], candidate_labels)
intent = intent_result['labels'][0]

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# 3) TOPIC DISTRIBUTION (DL METHOD)

model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embedding = model.encode([text])

topics = ["technology", "aviation", "policy"]
topic_embeddings = model.encode(topics)

similarities = cosine_similarity(doc_embedding, topic_embeddings)[0]

topic_scores = dict(zip(topics, similarities))

# Normalize to %
total = sum(topic_scores.values())
topic_percent = {k: round((v/total)*100, 2) for k,v in topic_scores.items()}

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# 4) LLM SUMMARY

summary = f"""
The article primarily discusses an aviation disaster involving an Air India Boeing aircraft.
Sentiment analysis indicates a {sent_label.lower()} tone, reflecting the tragic nature of the incident.
Emotion analysis highlights dominant feelings of {top_emotion}, driven by the loss of lives and ongoing investigation.
The intent of the article is mainly {intent}, as it provides factual reporting of the crash and investigation findings.
Topic modeling shows that the article is mostly related to aviation ({topic_percent['aviation']}%), followed by technology ({topic_percent['technology']}%) and policy ({topic_percent['policy']}%).
Overall, both LLM-based classification and embedding-based deep learning methods align in identifying the article as a factual report on an aviation-related tragedy with technical and regulatory implications.
"""

In [8]:
# 5) OUTPUT
print("\n RESULTS ")

print("\n1) SENTIMENT:")
print(sent_scores)
print("Dominant:", sent_label)

print("\n2) INTENT:")
print(intent)

print("\n3) EMOTION:")
print("Top Emotion:", top_emotion)

print("\n4) TOPIC DISTRIBUTION (%):")
print(topic_percent)

print("\n5) FINAL SUMMARY:")
print(summary)


 RESULTS 

1) SENTIMENT:
{'POSITIVE': np.float64(0.9481701254844666), 'NEGATIVE': np.float64(0.9965735276540121)}
Dominant: NEGATIVE

2) INTENT:
reporting

3) EMOTION:
Top Emotion: sadness

4) TOPIC DISTRIBUTION (%):
{'technology': np.float32(20.52), 'aviation': np.float32(67.17), 'policy': np.float32(12.31)}

5) FINAL SUMMARY:

The article primarily discusses an aviation disaster involving an Air India Boeing aircraft. 
Sentiment analysis indicates a negative tone, reflecting the tragic nature of the incident. 
Emotion analysis highlights dominant feelings of sadness, driven by the loss of lives and ongoing investigation. 
The intent of the article is mainly reporting, as it provides factual reporting of the crash and investigation findings. 
Topic modeling shows that the article is mostly related to aviation (67.16999816894531%), followed by technology (20.520000457763672%) and policy (12.3100004196167%). 
Overall, both LLM-based classification and embedding-based deep learning meth

The article is a factual report with negative tone covering a significant aviation accident. The analysis of emotions beneathlines sadness and fear, as there were many casualties. The objective is evidently informational / reporting as it does not concern any opinion but is concerned with the details of the investigation. The topic modeling reveals that the article focuses more on aviation and a focus on technology aircraft systems and policy investigation and regulation is secondary. LLM-based and deep learning approaches generate similar and complementary insights.